In [1]:
#Useful Imports
import torch as t
from torch import Tensor
import einops
from functools import partial
from tqdm import tqdm

from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    ActivationCache,
    FactoredMatrix,
    utils,
)
from transformer_lens.hook_points import HookPoint

from sae_lens import SAE, HookedSAETransformer

from jaxtyping import Float, Int, Bool

import circuitsvis as cv
import plotly.express as px

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#Login to HF
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load variables from .env
load_dotenv()

# Get the token (assuming it's named HF_TOKEN in your .env)
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    # login() will store the token in your local cache (~/.cache/huggingface)
    login(token=hf_token)
else:
    print("Error: HF_TOKEN not found in .env file.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Device selection
device = t.device(
    "mps" if t.backends.mps.is_available()
    else "cuda" if t.cuda.is_available()
    else "cpu"
)
print("device selected:",device,"-",t.cuda.get_device_name(0))

device selected: cuda - NVIDIA A100-SXM4-80GB


In [4]:
# Model Loading
model = HookedTransformer.from_pretrained(
    "gpt2-small",
    center_unembed=True,      # Center W_U
    center_writing_weights=True,
    fold_ln=True,             # Fold LayerNorm
    refactor_factored_attn_matrices=True,
    device=device
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2992.72it/s]


Loaded pretrained model gpt2-small into HookedTransformer


In [ ]:
# Generation utils 
def generate_sequence(model, prompt, max_tokens=10,insert_bos=False):
    # 1. Start by encoding the prompt into token IDs [cite: 86]
    # Current shape: [1, seq_len]
    current_tokens = model.to_tokens(prompt,prepend_bos=insert_bos) 
    
    # Llama chat models typically use <|eot_id|> as eos_token_id.
    eot_token_id = model.tokenizer.eos_token_id
    if eot_token_id is None:
        try:
            eot_token_id = model.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        except Exception:
            eot_token_id = None
    
    for _ in range(max_tokens):
        # 2. Forward pass to get logits 
        # We only care about the logits for the last position [cite: 354, 355]
        with t.inference_mode(): # Use inference mode for efficiency 
            logits = model(current_tokens) 
        
        # 3. Find the most likely next token (Greedy Search) 
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        
        # 4. Append the new token to the sequence [cite: 307]
        current_tokens = t.cat([current_tokens, next_token], dim=-1)
        
        # 5. Stop early if EOT token is generated
        if eot_token_id is not None and (next_token == eot_token_id).all():
            break
    
    # 6. Convert the final token ID list back to a string [cite: 93]
    return model.to_string(current_tokens[0])

In [ ]:
#Applying Chat Template
messages = [
    {"role": "system", "content": "You are a helpful mechanistic interpretability assistant."},
    {"role": "user", "content": "Explain the OV circuit."},
]

# add_generation_prompt=True adds the header for the assistant's turn
prompt = surrogate_model.tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)
print(prompt)